In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
# Load AID 720634 - skip first 5 metadata rows
casepase_raw = pd.read_csv('/Users/yaminichitikela/Documents/casepase/Casepase-activation/Data/tox21_caspase/AID_1346980_datatable.csv', 
                       skiprows=[1,2,3,4,5], 
                       low_memory=False)

print("AID 1346981 - casepase HepG2")
print(f"Loaded shape: {casepase_raw.shape}")

# Remove rows with missing CID
casepase_raw['PUBCHEM_CID'] = pd.to_numeric(casepase_raw['PUBCHEM_CID'], errors='coerce')
print(f"Rows with missing CID: {casepase_raw['PUBCHEM_CID'].isna().sum()}")
casepase_raw = casepase_raw.dropna(subset=['PUBCHEM_CID'])
casepase_raw['PUBCHEM_CID'] = casepase_raw['PUBCHEM_CID'].astype(int)
casepase_raw['PUBCHEM_ACTIVITY_SCORE'] = pd.to_numeric(casepase_raw['PUBCHEM_ACTIVITY_SCORE'], errors='coerce')

print(f"After cleanup: {casepase_raw.shape}")
print(f"Unique CIDs: {casepase_raw['PUBCHEM_CID'].nunique()}")

AID 1346981 - casepase HepG2
Loaded shape: (9667, 2049)
Rows with missing CID: 143
After cleanup: (9524, 2049)
Unique CIDs: 7671


In [56]:
# Remove duplicates within mito: keep highest score per CID+outcome
mito_sorted = casepase_raw.sort_values(
    by=['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_SCORE'],
    ascending=[True, False]
)

mito_clean = mito_sorted.drop_duplicates(
    subset=['PUBCHEM_CID'],
    keep='first'
).copy()

print(f"After dedup: {casepase_raw.shape} -> {mito_clean.shape}")
print(f"Removed {len(casepase_raw) - len(mito_clean)} duplicate rows")

print(mito_clean.head())


# Verify: Check CID 6
print("Verification - CID 4:")
print("\nBefore dedup (combined):")
display(casepase_raw[casepase_raw['PUBCHEM_CID'] == 403][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

print("\nAfter dedup (merged):")
display(mito_clean[mito_clean['PUBCHEM_CID'] == 403][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

After dedup: (9524, 2049) -> (7671, 2049)
Removed 1853 duplicate rows
      PUBCHEM_RESULT_TAG  PUBCHEM_SID  PUBCHEM_CID  \
5008                5009    144212021            4   
2141                2142    144209154            6   
2651                2652    144209664           11   
1496                1497    144208509           13   
4456                4457    144211469           33   

                  PUBCHEM_EXT_DATASOURCE_SMILES PUBCHEM_ACTIVITY_OUTCOME  \
5008                                    CC(CN)O                 Inactive   
2141  C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl                 Inactive   
2651                                   C(CCl)Cl                 Inactive   
1496                      C1=CC(=C(C=C1Cl)Cl)Cl                 Inactive   
4456                                   C(C=O)Cl                 Inactive   

      PUBCHEM_ACTIVITY_SCORE  \
5008                       0   
2141                       0   
2651                       0   
1496                

,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
1215,403,Inconclusive,10
9290,403,Inactive,0



After dedup (merged):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
1215,403,Inconclusive,10


In [57]:
# Save
mito_clean.to_csv('AID_1346980_cleaned.csv', index=False)

In [9]:
casepase_raw.tail(2)

,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Phenotype-Replicate_1,Potency-Replicate_1,...,Activity at 18.95 uM-Replicate_51,Activity at 42.46 uM-Replicate_51,Activity at 92.41 uM-Replicate_51,Activity at 240.3 uM-Replicate_51,Activity at 872.4 uM-Replicate_51,Activity at 1849.0 uM-Replicate_51,Activity at 4135.0 uM-Replicate_51,Activity at 9246.0 uM-Replicate_51,Activity at 20670.0 uM-Replicate_51,Activity at 46230.0 uM-Replicate_51
9664,9665,251919989,13059052,CC(C)(C1=CC(=C(C=C1)O)C2=CC=CC=C2)C3=CC(=C(C=C...,Inconclusive,10,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,61.1306,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9665,9666,251919990,14178379,C1=CC=C(C=C1)COC2=CC=C(C=C2)S(=O)(=O)C3=CC=CC=C3O,Inactive,0,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Inactive,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
casepase_raw.sort_values(by=['PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_CID'], ascending=[False, True], inplace=True)





,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Phenotype-Replicate_1,Potency-Replicate_1,...,Activity at 18.95 uM-Replicate_51,Activity at 42.46 uM-Replicate_51,Activity at 92.41 uM-Replicate_51,Activity at 240.3 uM-Replicate_51,Activity at 872.4 uM-Replicate_51,Activity at 1849.0 uM-Replicate_51,Activity at 4135.0 uM-Replicate_51,Activity at 9246.0 uM-Replicate_51,Activity at 20670.0 uM-Replicate_51,Activity at 46230.0 uM-Replicate_51
5545,5546,144212558,9786,C1=CC=C(C=C1)[Sn](C2=CC=CC=C2)(C3=CC=CC=C3)F,Active,90,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.0611,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2785,2786,144209798,16682804,CC(=O)O[Sn](C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3,Active,90,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.1220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2667,2668,144209680,457193,C[C@@H]1[C@@H](C(=O)N[C@@H](C(=O)N2CCC[C@H]2C(...,Active,89,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.1880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5026,5027,144212039,108956,C=CC(=O)OCC(CO)(COCC(COC(=O)C=C)(COC(=O)C=C)CO...,Active,88,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.4797,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4058,4059,144211071,6327657,C1=CC=C(C=C1)[Sn](C2=CC=CC=C2)C3=CC=CC=C3.O,Active,88,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.1369,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# get values greater than 5 activity score 
high_score = casepase_raw[casepase_raw['PUBCHEM_ACTIVITY_SCORE'] > 5]


,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Phenotype-Replicate_1,Potency-Replicate_1,...,Activity at 18.95 uM-Replicate_51,Activity at 42.46 uM-Replicate_51,Activity at 92.41 uM-Replicate_51,Activity at 240.3 uM-Replicate_51,Activity at 872.4 uM-Replicate_51,Activity at 1849.0 uM-Replicate_51,Activity at 4135.0 uM-Replicate_51,Activity at 9246.0 uM-Replicate_51,Activity at 20670.0 uM-Replicate_51,Activity at 46230.0 uM-Replicate_51
5545,5546,144212558,9786,C1=CC=C(C=C1)[Sn](C2=CC=CC=C2)(C3=CC=CC=C3)F,Active,90,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.0611,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2785,2786,144209798,16682804,CC(=O)O[Sn](C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3,Active,90,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.1220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2667,2668,144209680,457193,C[C@@H]1[C@@H](C(=O)N[C@@H](C(=O)N2CCC[C@H]2C(...,Active,89,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.1880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5026,5027,144212039,108956,C=CC(=O)OCC(CO)(COCC(COC(=O)C=C)(COC(=O)C=C)CO...,Active,88,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.4797,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4058,4059,144211071,6327657,C1=CC=C(C=C1)[Sn](C2=CC=CC=C2)C3=CC=CC=C3.O,Active,88,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,0.1369,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3531,3532,144210544,53316396,C1=CC(=CC(=C1)C2=N[N-]C(=N2)C(=O)N)C3=C(C=CC(=...,Inconclusive,10,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,68.5896,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137,138,144206852,60196351,C1[C@@H](C2=C(CN1)C(=C(C=C2)O)O)C3=CC(=C(C=C3)...,Inconclusive,10,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,33.4915,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3537,3538,144210550,60196403,COC1=CC=CC(=C1)C2=C(C=C(C=C2)/C=C\CN3CCCCCC3)C...,Inconclusive,10,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,60.8535,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3561,3562,144210574,60196407,CCOC(=O)O[C@H](C)OC(=O)C1=CC2=C(C=C1)N(C(=C2)C...,Inconclusive,10,http://assay.nih.gov/htsws/rest/display/tox21-...,NaN,Activator,14.3460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
